### Installation

In [1]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

  Using cached unsloth-2026.9.4-py3-none-any.whl.metadata (75 kB)
  Using cached unsloth_zoo-2026.9.3-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.12.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached tyro-1.0.16-py3-none-any.whl.metadata (12 kB)
  Using cached protobuf-7.36.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached triton-3.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (978 bytes)
  Using cached sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (33 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-many

### Unsloth

In [3]:
from unsloth import FastSentenceTransformer
 
fourbit_models = [
    "unsloth/all-MiniLM-L6-v2",
    "unsloth/embeddinggemma-300m",
    "unsloth/Qwen3-Embedding-4B",
    "unsloth/Qwen3-Embedding-0.6B",
    "unsloth/all-mpnet-base-v2",
    "unsloth/gte-modernbert-base",
    "unsloth/bge-m3"
 
] # More models at https://huggingface.co/unsloth
 

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
model = FastSentenceTransformer.from_pretrained(
    model_name = "unsloth/all-MiniLM-L6-v2",
    max_seq_length = 512,   # Choose any for long context!
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastSentenceTransformer.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ['value', 'key', 'dense', 'query'],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = False, # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type = "FEATURE_EXTRACTION"
)

Unsloth: torch.compile will be applied automatically if max_steps > 2616


<a name="Data"></a>
### Data Prep
We now use the `sentence-transformers/all-nli` dataset. This dataset provides a massive corpus of sentence pairs labeled for entailment, neutral, or contradiction. It is widely used to fine-tune models for generating semantically meaningful sentence embeddings.

In [ ]:
import ast
import random
import torch
import pandas as pd
from datasets import Dataset
 
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
 
# %%
df = pd.read_csv("hf://datasets/regularpooria/CVE_CWE_Software_Mapping_Dataset/Global_Dataset.csv")
 
# %%
def parse_parents(parents_raw):
    """
    PARENTS is stored as a stringified list of (cwe_id, description) tuples,
    e.g. "[('CWE-389', 'This category includes...'), ('CWE-137', '...')]".
    Safely parse it back into a Python list of tuples.
    """
    if parents_raw is None or (isinstance(parents_raw, float) and pd.isna(parents_raw)):
        return []
    if isinstance(parents_raw, list):
        return parents_raw
    try:
        parsed = ast.literal_eval(parents_raw)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return []
 
 
def build_cwe_lookup(df: pd.DataFrame):
    """
    Build:
      cwe_to_desc: {cwe_id: cwe_description}
      cwe_to_parent_ids: {cwe_id: set(parent_cwe_ids)}
      all_parent_pairs: {parent_cwe_id: parent_description}, used as a pool
                         for "taxonomy-flavored" negatives.
    """
    cwe_to_desc = (
        df.dropna(subset=["CWE-ID", "CWE-DESCRIPTION"])
        .drop_duplicates(subset=["CWE-ID"])
        .set_index("CWE-ID")["CWE-DESCRIPTION"]
        .to_dict()
    )
 
    cwe_to_parent_ids = {}
    all_parent_pairs = {}
 
    for _, row in df.iterrows():
        cwe_id = row["CWE-ID"]
        parents = parse_parents(row.get("PARENTS"))
        parent_ids = set()
        for parent_id, parent_desc in parents:
            parent_ids.add(parent_id)
            all_parent_pairs[parent_id] = parent_desc
        if cwe_id in cwe_to_parent_ids:
            cwe_to_parent_ids[cwe_id] |= parent_ids
        else:
            cwe_to_parent_ids[cwe_id] = parent_ids
 
    return cwe_to_desc, cwe_to_parent_ids, all_parent_pairs
 
 
def build_pairs(
    df: pd.DataFrame,
    n_hard_negatives: int = 1,
    n_random_negatives: int = 1,
    include_parent_positives: bool = True,
) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
        cve_id, cve_text, cwe_text, label, pair_type
    where pair_type is one of:
        "positive_own_cwe"
        "positive_parent_cwe"
        "negative_random_cwe"
        "negative_taxonomy_cwe"
    """
    df = df.dropna(subset=["CVE-DESCRIPTION-NORMALIZED", "CWE-ID", "CWE-DESCRIPTION"]).reset_index(drop=True)
 
    cwe_to_desc, cwe_to_parent_ids, all_parent_pairs = build_cwe_lookup(df)
    all_cwe_ids = list(cwe_to_desc.keys())
    all_parent_ids = list(all_parent_pairs.keys())
 
    rows = []
 
    for _, row in df.iterrows():
        cve_id = row["CVE-ID"]
        cve_text = row["CVE-DESCRIPTION-NORMALIZED"]
        own_cwe_id = row["CWE-ID"]
        own_cwe_desc = row["CWE-DESCRIPTION"]
 
        # ---- Positive #1: the CVE's actual CWE description ----
        rows.append(
            {
                "cve_id": cve_id,
                "cve_text": cve_text,
                "cwe_text": own_cwe_desc,
                "label": 1,
                "pair_type": "positive_own_cwe",
            }
        )
 
        # ---- Positive #2 (optional): the CVE's parent-CWE description(s) ----
        if include_parent_positives:
            parents = parse_parents(row.get("PARENTS"))
            for parent_id, parent_desc in parents:
                rows.append(
                    {
                        "cve_id": cve_id,
                        "cve_text": cve_text,
                        "cwe_text": parent_desc,
                        "label": 1,
                        "pair_type": "positive_parent_cwe",
                    }
                )
 
        own_parent_ids = cwe_to_parent_ids.get(own_cwe_id, set())
 
        # ---- Negative #1: random, unrelated CWE ----
        candidates = [c for c in all_cwe_ids if c != own_cwe_id and c not in own_parent_ids]
        n_take = min(n_random_negatives, len(candidates))
        for neg_cwe_id in random.sample(candidates, n_take):
            rows.append(
                {
                    "cve_id": cve_id,
                    "cve_text": cve_text,
                    "cwe_text": cwe_to_desc[neg_cwe_id],
                    "label": 0,
                    "pair_type": "negative_random_cwe",
                }
            )
 
        # ---- Negative #2 (harder): a parent-level CWE description that
        #      belongs to the taxonomy but is NOT actually a parent of this CVE's CWE ----
        hard_candidates = [
            pid for pid in all_parent_ids if pid != own_cwe_id and pid not in own_parent_ids
        ]
        n_take_hard = min(n_hard_negatives, len(hard_candidates))
        for neg_parent_id in random.sample(hard_candidates, n_take_hard):
            rows.append(
                {
                    "cve_id": cve_id,
                    "cve_text": cve_text,
                    "cwe_text": all_parent_pairs[neg_parent_id],
                    "label": 0,
                    "pair_type": "negative_taxonomy_cwe",
                }
            )
 
    pairs_df = pd.DataFrame(rows)
    return pairs_df
 
 
def build_triplet_dataset(pairs_df: pd.DataFrame) -> Dataset:
    """
    Reshape (cve_text, cwe_text, label, pair_type) rows into (anchor, positive,
    negative) triplets for MultipleNegativesRankingLoss. Prefers the hard
    taxonomy negative over the random negative when both exist.
    """
    triplets = []
    for cve_id, group in pairs_df.groupby("cve_id"):
        pos_row = group[group["pair_type"] == "positive_own_cwe"].iloc[0]
 
        neg_candidates = group[group["pair_type"] == "negative_taxonomy_cwe"]
        if neg_candidates.empty:
            neg_candidates = group[group["pair_type"] == "negative_random_cwe"]
        neg_row = neg_candidates.iloc[0]
 
        triplets.append(
            {
                "anchor": pos_row["cve_text"],
                "positive": pos_row["cwe_text"],
                "negative": neg_row["cwe_text"],
            }
        )
 
    return Dataset.from_list(triplets)

In [5]:
print(f"Loaded {len(df):,} rows. Columns: {list(df.columns)}")
 
print("Building positive/negative pairs...")
pairs_df = build_pairs(
    df,
    n_hard_negatives=1,
    n_random_negatives=1,
    include_parent_positives=True,
)
print(f"\nBuilt {len(pairs_df):,} pairs.")
print(pairs_df["pair_type"].value_counts())
print(f"\nLabel balance:\n{pairs_df['label'].value_counts(normalize=True)}")

out_path = "cve_cwe_matching_pairs.csv"
pairs_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

print("\nSample rows:")
print(pairs_df.sample(5, random_state=RANDOM_SEED).to_string(index=False))

Loaded 60,897 rows. Columns: ['ID', 'CVE-ID', 'CVSS-V3', 'CVSS-V2', 'SEVERITY', 'CVE-DESCRIPTION', 'CWE-ID', 'YEAR', 'IS-IN-CWE699', 'PARENTS', 'CWE-DESCRIPTION', 'CVE-DESCRIPTION-NORMALIZED']
Building positive/negative pairs...

Built 245,861 pairs.
pair_type
positive_parent_cwe      63170
positive_own_cwe         60897
negative_random_cwe      60897
negative_taxonomy_cwe    60897
Name: count, dtype: int64

Label balance:
label
1    0.504623
0    0.495377
Name: proportion, dtype: float64

Saved to cve_cwe_matching_pairs.csv

Sample rows:
        cve_id                                                                                                                                                                                                                                                                                                                                                                                                                                                         

Let's take a look at the dataset structure:

## Baseline Inference
Let's test the base model before fine-tuning to see how it performs on our specific domain.

In [ ]:
def test_inference_cve_cwe(model, pairs_df, run_name="Run", n_examples=1):
    """
    Pick a CVE description as the query, and score it against its true CWE
    description plus a few unrelated ones, using plain cosine similarity.
    """
    sample_cve_ids = pairs_df["cve_id"].drop_duplicates().sample(
        n_examples, random_state=42
    )
 
    for cve_id in sample_cve_ids:
        group = pairs_df[pairs_df["cve_id"] == cve_id]
        query = group.iloc[0]["cve_text"]
 
        candidates, labels = [], []
        for _, row in group.iterrows():
            candidates.append(row["cwe_text"])
            labels.append(row["label"])
 
        query_emb = model.encode(query, convert_to_tensor=True)
        candidate_embs = model.encode(candidates, convert_to_tensor=True)
        scores = torch.nn.functional.cosine_similarity(
            query_emb.unsqueeze(0), candidate_embs
        )
 
        results = list(zip(candidates, scores.tolist(), labels))
        results.sort(key=lambda x: x[1], reverse=True)
 
        print(f"\n--- {run_name} | {cve_id} ---")
        print(f"Query: {query[:150]}...")
        for text, score, label in results:
            tag = "POS" if label == 1 else "NEG"
            print(f"{score:.4f} | {tag} | {text[:100]}")
 
test_inference_cve_cwe(model, pairs_df, run_name="Pre-Training (CVE/CWE)")
 

Loading base model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2870.06it/s]


NameError: name 'util' is not defined

<a name="Train"></a>
### Train the model
Now let's train our model. We use `MultipleNegativesRankingLoss`

 This loss function uses other positives in the same batch as negative examples, which is efficient for contrastive learning.

In [ ]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses
)
from sentence_transformers.training_args import BatchSamplers
from unsloth import is_bf16_supported
loss = losses.MultipleNegativesRankingLoss(model)
 
trainer = SentenceTransformerTrainer(
    model = model,
    train_dataset = dataset,
    loss = loss,
    args = SentenceTransformerTrainingArguments(
        output_dir = "output",
        num_train_epochs = 2,
        per_device_train_batch_size = 256,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 50,
        warmup_ratio = 0.03,
        report_to = "none",
        lr_scheduler_type = 'linear',
        # Because we have duplicate anchors in the dataset, we don't want
        # to accidentally use them for negative examples
        batch_sampler = BatchSamplers.NO_DUPLICATES,
    ),
 
)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")
 

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")
 

230.1043 seconds used for training.
3.84 minutes used for training.
Peak reserved memory = 3.287 GB.
Peak reserved memory for training = 3.209 GB.
Peak reserved memory % of max memory = 22.298 %.
Peak reserved memory for training % of max memory = 21.769 %.


<a name="Inference"></a>
### Inference
Let's run the model after training to see the improvements!

In [ ]:
test_inference_cve_cwe(model, pairs_df, run_name="Post-Training (CVE/CWE)")



--- Post-Training Results for query: 'A soccer player in a blue jersey is running across the field.' ---
0.4459 | An athlete is practicing a sport outdoors.
0.4322 | The player is about to score a goal for his team.
0.3113 | Jersey is a knit fabric used predominantly for clothing manufacture.
0.0944 | A person is sitting quietly on the grass.

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit, scroll down!

In [ ]:
model.save_pretrained("all_minilm_l6_v2_lora")  # Local saving
model.tokenizer.save_pretrained("all_minilm_l6_v2_lora")
# model.push_to_hub("your_name/all_minilm_l6_v2_lora", token = "YOUR_HF_TOKEN") # Online saving
# model.tokenizer.push_to_hub("your_name/all_minilm_l6_v2_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastSentenceTransformer
    model = FastSentenceTransformer.from_pretrained(
        "model",
        # for_inference = True
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("all_minilm_l6_v2_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/all_minilm_l6_v2_finetune_16bit", tokenizer = model.tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("all_minilm_l6_v2_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/all_minilm_l6_v2_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("all_minilm_l6_v2_finetune",)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/all_minilm_l6_v2_finetune", token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("all_minilm_l6_v2_finetune", quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/all_minilm_l6_v2_finetune", quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("all_minilm_l6_v2_finetune", quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/all_minilm_l6_v2_finetune", quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/all_minilm_l6_v2_finetune", # Change hf to your username!
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>